# 8.3 Linear Discriminant Analysis (LDA)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 8: Dimensionality Reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-08-dimensionality-reduction/08.03_linear-discriminant-analysis.ipynb)

Companion code for Section 8.3 of the book. Run the cells in order from top to bottom.

**Example 8.3.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# ── LDA from scratch ──────────────────────────────────────────────
class LDAScratch:
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.scalings_    = None  # Discriminant directions
        self.means_       = None  # Class means
        self.priors_      = None  # Class priors

    def fit(self, X, y):
        classes = np.unique(y)
        n, d    = X.shape
        n_classes = len(classes)

        # Overall mean
        mu = X.mean(axis=0)

        # Class means and priors
        self.means_  = {c: X[y==c].mean(axis=0) for c in classes}
        self.priors_ = {c: (y==c).sum()/n        for c in classes}

        # Between-class scatter Sᴮ
        S_B = np.zeros((d, d))
        for c in classes:
            nc   = (y==c).sum()
            diff = (self.means_[c] - mu).reshape(-1,1)
            S_B += nc * (diff @ diff.T)

        # Within-class scatter Sᵂ
        S_W = np.zeros((d, d))
        for c in classes:
            X_c  = X[y==c] - self.means_[c]
            S_W += X_c.T @ X_c

        # Solve Sᵂ⁻¹Sᴮ w = λw
        # Add small regularisation for numerical stability
        S_W_reg = S_W + 1e-6 * np.eye(d)
        A = np.linalg.inv(S_W_reg) @ S_B
        eigenvalues, eigenvectors = np.linalg.eigh(A)

        # Sort descending
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues  = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Keep only real parts and positive eigenvalues
        k = self.n_components or min(n_classes - 1, d)
        self.scalings_ = eigenvectors[:, :k].real
        self.eigenvalues_ = eigenvalues[:k].real

        return self

    def transform(self, X):
        return X @ self.scalings_

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


# ── Compare PCA vs LDA on Iris (3 classes → max 2 LDA components) ─
for dataset_name, dataset in [('Iris (3 classes)', load_iris()),
                                ('Wine (3 classes)', load_wine())]:
    X, y = dataset.data, dataset.target
    X_sc = StandardScaler().fit_transform(X)
    n_classes = len(np.unique(y))

    print(f'\n=== {dataset_name}: {X.shape[1]} features, {n_classes} classes ===')

    # LDA (from scratch)
    lda_sc = LDAScratch(n_components=2)
    Z_lda_sc = lda_sc.fit_transform(X_sc, y)

    # LDA (sklearn)
    lda = LinearDiscriminantAnalysis(n_components=2)
    Z_lda = lda.fit_transform(X_sc, y)

    # PCA for comparison
    pca = PCA(n_components=2)
    Z_pca = pca.fit_transform(X_sc)

    print(f'  LDA discriminability ratio (eigenvalues): {lda_sc.eigenvalues_.round(4)}')

    # Classification performance
    for method, Z in [('PCA (2D)', Z_pca), ('LDA (2D)', Z_lda)]:
        lr = LogisticRegression(max_iter=1000)
        scores = cross_val_score(lr, Z, y, cv=5, scoring='accuracy')
        print(f'  {method:12s}: CV accuracy = {scores.mean():.4f} ± {scores.std():.4f}')


# ── Visualise PCA vs LDA side by side ────────────────────────────
iris = load_iris()
X_sc = StandardScaler().fit_transform(iris.data)
y    = iris.target
names = iris.target_names

pca = PCA(n_components=2)
Z_pca = pca.fit_transform(X_sc)

lda = LinearDiscriminantAnalysis(n_components=2)
Z_lda = lda.fit_transform(X_sc, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('PCA vs LDA - Iris Dataset (4 features → 2D)',
            fontsize=14, fontweight='bold')

palette = ['#E53935','#1976D2','#43A047']
for ax, Z, method, pc_labels in [
    (axes[0], Z_pca, 'PCA', ['PC1 (Unsupervised)','PC2']),
    (axes[1], Z_lda, 'LDA', ['LD1 (Supervised)', 'LD2']),
]:
    for k in range(3):
        mask = y == k
        ax.scatter(Z[mask,0], Z[mask,1], s=40, alpha=0.8,
                  color=palette[k], label=names[k], edgecolors='white', lw=0.5)

    ax.set_xlabel(pc_labels[0], fontsize=12)
    ax.set_ylabel(pc_labels[1], fontsize=12)
    ax.set_title(f'{method} - Classes {'separable' if method=="LDA" else "partially overlapping"}',
               fontsize=12)
    ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pca_vs_lda.png', dpi=150, bbox_inches='tight')
plt.show()

**Example 8.3.2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits, load_breast_cancer
import time
import warnings; warnings.filterwarnings('ignore')

# ── t-SNE on Digits dataset (10 classes, 64 features) ────────────
digits = load_digits()
X_digits = digits.data      # 1797 × 64 (8×8 pixel images)
y_digits = digits.target    # 0-9 digit labels

# Always apply PCA first to reduce noise and speed up t-SNE
X_pca50 = PCA(n_components=50).fit_transform(X_digits)
print(f'PCA: 64 → 50 dimensions (preserving {PCA(50).fit(X_digits).explained_variance_ratio_.sum():.1%} variance)')

# ── Effect of perplexity ─────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('t-SNE Analysis - Digits Dataset (1797 samples, 10 classes)',
            fontsize=14, fontweight='bold')

palette_10 = plt.cm.tab10(np.linspace(0,1,10))

# Different perplexities
perplexities = [5, 15, 30, 50]
for idx, perp in enumerate(perplexities):
    ax = axes[0, idx]
    t0 = time.time()
    tsne = TSNE(n_components=2, perplexity=perp, n_iter=1000,
               random_state=42, method='barnes_hut')
    Z_tsne = tsne.fit_transform(X_pca50)
    t1 = time.time()

    for k in range(10):
        mask = y_digits == k
        ax.scatter(Z_tsne[mask,0], Z_tsne[mask,1], s=8, alpha=0.8,
                  color=palette_10[k], label=str(k))

    ax.set_title(f'Perplexity={perp}  ({t1-t0:.1f}s)', fontsize=11)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    if idx == 0: ax.legend(title='Digit', fontsize=7, ncol=2, markerscale=2)
    ax.grid(alpha=0.2)

# ── Best t-SNE (perplexity=30) with annotations ──────────────────
tsne_best = TSNE(n_components=2, perplexity=30, n_iter=2000, random_state=42)
Z_best = tsne_best.fit_transform(X_pca50)

ax = axes[1, 0]
sc = ax.scatter(Z_best[:,0], Z_best[:,1], c=y_digits,
               cmap='tab10', s=10, alpha=0.8)
plt.colorbar(sc, ax=ax, label='Digit')
ax.set_title('Best t-SNE (perplexity=30)\nAll 10 digits cleanly separated')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2'); ax.grid(alpha=0.2)

# Annotate cluster centres
for k in range(10):
    mask = y_digits == k
    cx, cy = Z_best[mask,0].mean(), Z_best[mask,1].mean()
    ax.annotate(str(k), (cx, cy), fontsize=14, fontweight='bold',
               ha='center', va='center',
               bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

# ── PCA vs t-SNE comparison ───────────────────────────────────────
ax = axes[1, 1]
Z_pca2 = PCA(n_components=2).fit_transform(X_digits)
sc2 = ax.scatter(Z_pca2[:,0], Z_pca2[:,1], c=y_digits, cmap='tab10', s=8, alpha=0.7)
ax.set_title('PCA 2D (overlapping, hard to read)'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.grid(alpha=0.2)

# ── KL divergence loss curve ─────────────────────────────────────
ax = axes[1, 2]
# Run t-SNE tracking KL divergence
kl_divs = []
for n_iter in [250, 500, 750, 1000, 1500, 2000]:
    tsne_iter = TSNE(n_components=2, perplexity=30, n_iter=n_iter, random_state=42)
    tsne_iter.fit_transform(X_pca50[:300])  # Subset for speed
    kl_divs.append(tsne_iter.kl_divergence_)

ax.plot([250,500,750,1000,1500,2000], kl_divs, 'b-o', ms=8, lw=2.5)
ax.set_xlabel('Iterations'); ax.set_ylabel('KL Divergence')
ax.set_title('t-SNE Convergence\n(lower KL = better embedding)')
ax.grid(alpha=0.3)

# ── Breast cancer t-SNE ──────────────────────────────────────────
ax = axes[1, 3]
bc_data = load_breast_cancer()
X_bc_sc = StandardScaler().fit_transform(bc_data.data)
tsne_bc = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
Z_bc = tsne_bc.fit_transform(X_bc_sc)
colors = ['#E53935' if c==0 else '#1976D2' for c in bc_data.target]
ax.scatter(Z_bc[:,0], Z_bc[:,1], c=colors, s=15, alpha=0.8, edgecolors='none')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#E53935',label='Malignant'),
                   Patch(color='#1976D2',label='Benign')], fontsize=10)
ax.set_title('Breast Cancer t-SNE\n(clear separation of 2 classes)')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2'); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('tsne_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nt-SNE KL divergences at different iterations:')
for it, kl in zip([250,500,750,1000,1500,2000], kl_divs):
    print(f'  iter={it:>5}: KL={kl:.4f}')